# 08 — theme (Chakra v3)

The pre-render gate for the v3 migration. It answers one question:
does every token, recipe and variant a call site can reach still resolve
to what it resolved to under v2?

This matters because v3 fails **silently** in exactly the place a migration
is most likely to slip. An unresolvable token path is not an error: it is
emitted as the literal string, producing a well-formed CSS declaration full
of garbage. `tsc` is happy, `gatsby build` is happy, and the colour is wrong.
No screenshot diff catches it either, if the wrong colour happens to be
close.

The probe is `support/dump-theme-facts.ts`, which has to be TypeScript
because a Chakra system only exists once its config has been evaluated. It
decides nothing; it prints. Everything below judges.


In [ ]:
import jaen_testkit as k
k.start_run('08-theme')
print(k.CONFIG['repo_root'])


In [ ]:
import json

# ts-node has to be told to ignore the repo's own moduleResolution:'bundler',
# which cannot combine with the commonjs module output node needs here.
TS_OPTS = ('{"module":"commonjs","moduleResolution":"node","target":"es2020",'
           '"esModuleInterop":true,"skipLibCheck":true,"strict":false}')

FACTS = None
with k.section('probe'):
    with k.check('theme facts dump runs') as c:
        r = c.require(k.sh(
            "node -r ts-node/register/transpile-only tests/support/dump-theme-facts.ts",
            cwd=k.CONFIG['repo_root'],
            env={'TS_NODE_COMPILER_OPTIONS': TS_OPTS},
            timeout=180))
        FACTS = json.loads(r.text)
        c.expect_true(isinstance(FACTS, dict), 'parsed %d top-level keys' % len(FACTS))


## System shape

Two settings carry the whole isolation design.

`cssVarsPrefix: 'jaen'` replaces v2's `cssVarsRoot="#momo"`. The scoped root
cannot be carried forward: in v3 only the *base* bucket honours it, while the
dark bucket always lands on `.dark, .dark .chakra-theme:not(.light)`. With
`.dark` on `<html>` and `#momo` a descendant, the scoped block wins and dark
mode dies inside the CMS. Disjoint prefixes cannot collide whatever selector
they land on.

`disableLayers: true` is needed while `dist/jaen.css` still ships Tailwind's
**unlayered** preflight. An unlayered normal declaration beats every cascade
layer regardless of specificity, so with layers on, `h1..h6 {font-size:
inherit}` would flatten every Heading in both repos. Remove it once Tailwind
is gone, not before.


In [ ]:
with k.section('system shape'):
    cfg = FACTS['config']

    with k.check('variable prefix is jaen, not chakra') as c:
        c.expect_equal(cfg['cssVarsPrefix'], 'jaen')

    with k.check('cascade layers disabled while the Tailwind preflight ships') as c:
        c.expect_equal(cfg['disableLayers'], True)

    with k.check('jaen owns the reset (it is mounted on every route)') as c:
        c.expect_equal(cfg['preflight'], True)

    # v3 moved lg from 62em (992px) to 1024px. Every responsive array in both
    # repos was authored against 992, so the v2 value is pinned.
    with k.check('breakpoints pinned to v2 (lg stays 992px)') as c:
        c.expect_equal(cfg['breakpoints'], {
            'sm': '480px', 'md': '768px', 'lg': '992px',
            'xl': '1280px', '2xl': '1536px'})


## Token equivalence

Expected values are frozen from the v2 theme as it stood at the migration
(`packages/gatsby-plugin-jaen/src/theme/jaen-theme/foundations/`). They are
written out rather than read from the old files, because the old files are
deleted by the commit this notebook gates.


In [ ]:
# frozen from v2's foundations/{colors,sizes,space}.ts
V2_BASE = {
    'colors.gray.25': '#fcfdfe',
    'colors.gray.50': '#f4f8fa',
    'colors.gray.950': '#14151e',
    'sizes.11': '2.75rem',
    'sizes.15': '3.75rem',
    'spacing.4.5': '1.125rem',
}

with k.section('base tokens jaen overrides'):
    for path, expected in V2_BASE.items():
        with k.check('%s == %s' % (path, expected)) as c:
            c.expect_equal(FACTS['tokens'][path], expected)

    # v3 has no 3xs. v2 did, and checkbox's sm indicator used it, so the
    # recipe carries the literal 0.45rem instead of a path that resolves to
    # its own name.
    with k.check('fontSizes.3xs is known-missing, not silently used') as c:
        c.expect_equal(FACTS['tokens']['fontSizes.3xs'], None)
        c.note('checkbox sm indicator carries the literal 0.45rem')


## Semantic tokens: nested, not dotted

The trap this catches is subtle and would have survived every other gate.
v2 wrote all eighteen colour tokens as flat string keys (`'bg.canvas'`). In
v3 a dotted key is a *different token*: it is escaped into its own variable
`--jaen-colors-bg\.subtle`, sitting beside v3's built-in
`--jaen-colors-bg-subtle` rather than overriding it. Both get emitted,
`token('colors.bg.subtle')` returns the escaped one, and recipes disagree
about which they meant.


In [ ]:
SEMANTIC = [
    'colors.bg.canvas', 'colors.bg.surface', 'colors.bg.subtle',
    'colors.bg.muted', 'colors.bg.translucent',
    'colors.bg.accent.default', 'colors.bg.accent.subtle', 'colors.bg.accent.muted',
    'colors.fg.default', 'colors.fg.emphasized', 'colors.fg.muted',
    'colors.fg.subtle', 'colors.fg.inverted',
    'colors.fg.accent.default', 'colors.fg.accent.subtle', 'colors.fg.accent.muted',
    'colors.border.default', 'colors.border.emphasized', 'colors.border.active',
    'colors.accent', 'colors.success', 'colors.error',
    'shadows.xs', 'shadows.focus',
]

with k.section('semantic tokens'):
    with k.check('every semantic token resolves') as c:
        missing = [p for p in SEMANTIC if FACTS['tokens'].get(p) is None]
        c.expect_true(not missing, 'unresolved: %s' % missing)

    with k.check('no escaped (dotted-key) token names were emitted') as c:
        escaped = [v for v in FACTS['emittedVars']['jaen'] if '\\.' in v]
        c.expect_true(not escaped, 'escaped names: %s' % escaped[:5])


## The colorPalette contract

`colorPalette: 'brand'` rewrites every `colorPalette.*` reference in a recipe
to `brand.*`, but only for the eight slots v3 defines. A palette missing one
emits the literal token path as a CSS value.

`solidHover` and `solidActive` are jaen's own additions: v3's contract has no
slot one step past `solid`, and the CMS buttons need one.

This is also the replacement for `withDefaultColorScheme`, which v3 dropped
with no equivalent, and whose absence is invisible: every button would simply
turn grey while the build stayed green.


In [ ]:
PALETTE = ['solid', 'contrast', 'fg', 'subtle', 'muted', 'emphasized',
           'focusRing', 'border', 'solidHover', 'solidActive']

with k.section('colorPalette'):
    with k.check('brand fills all ten slots') as c:
        missing = [s for s in PALETTE
                   if FACTS['tokens'].get('colors.brand.%s' % s) is None]
        c.expect_true(not missing, 'missing: %s' % missing)

    with k.check('colorPalette.solid resolves through the palette var') as c:
        c.expect_contains(json.dumps(FACTS['css']['colorPaletteSolid']),
                          'color-palette-solid')

    with k.check('the /opacity suffix emits color-mix over a real var') as c:
        emitted = json.dumps(FACTS['css']['opacitySuffix'])
        c.expect_contains(emitted, 'color-mix')
        c.expect_contains(emitted, 'var(--jaen-colors-brand-400)')

    # Documents the silent-failure mode this whole notebook exists for.
    with k.check('an unresolvable path emits its own name (the silent failure)') as c:
        c.expect_contains(json.dumps(FACTS['css']['unresolvable']),
                          'components.nope.missing')


## Recipe coverage

A variant that does not exist is not an error either: the element falls back
to the base style. Invisible to `tsc`, invisible to the build.

The second check below is the proof that `mergeConfigs` *extends* v3's own
recipe rather than replacing it. That is what lets `button.ts` shrink from
411 lines to about 200: v2 spread `theme.components.Button.variants.ghost`
explicitly, and in v3 that spread becomes its absence.


In [ ]:
USED_BUTTON_VARIANTS = [
    'primary', 'secondary', 'secondary.subtle', 'text',
    'primary.accent', 'secondary.accent', 'text.accent',
    'ghost', 'outline',
    'field-highlighter-tooltip', 'field-highlighter-tooltip-text',
]
V3_OWN_BUTTON_VARIANTS = ['solid', 'subtle', 'surface', 'outline', 'ghost', 'plain']

with k.section('recipes'):
    with k.check('every button variant a call site uses exists') as c:
        have = FACTS['recipes']['button']['variant']
        missing = [v for v in USED_BUTTON_VARIANTS if v not in have]
        c.expect_true(not missing, 'missing: %s' % missing)

    with k.check("v3's own button variants survived the merge") as c:
        have = FACTS['recipes']['button']['variant']
        missing = [v for v in V3_OWN_BUTTON_VARIANTS if v not in have]
        c.expect_true(not missing, 'lost: %s' % missing)

    with k.check('all seven button sizes present') as c:
        c.expect_equal(sorted(FACTS['recipes']['button']['size']),
                       sorted(['2xs','xs','sm','md','lg','xl','2xl']))

    with k.check('link keeps underline, drops the unused menu variant') as c:
        have = FACTS['recipes']['link']['variant']
        c.expect_true('underline' in have, str(have))
        c.expect_true('menu' not in have, 'menu had no consumer')

    with k.check('table striped became a boolean variant') as c:
        c.expect_true('true' in FACTS['slotRecipes']['table']['striped'],
                      str(FACTS['slotRecipes']['table']))


## Heading sizes keep their v2 meaning

The size axis changed meaning between versions. v2 mapped `size="md"` to
`fontSize: 4xl`; v3 maps it to `textStyle md`, which is 1rem. Ported
verbatim, or all 38 headings in the CMS silently collapse to body size.


In [ ]:
V2_HEADING = {'2xl': '7xl', 'xl': '6xl', 'lg': '5xl',
              'md': '4xl', 'sm': '3xl', 'xs': '2xl'}

with k.section('heading sizes'):
    resolved = FACTS['recipes']['heading']['resolved']
    for size, font in V2_HEADING.items():
        with k.check('heading size=%s -> fontSize %s' % (size, font)) as c:
            c.expect_equal((resolved.get(size) or {}).get('fontSize'), font)


## Isolation

Both systems mount on a public route. Nothing jaen emits may land in the
site's namespace or the other way round.

`--chakra-empty` is the one allowance, and it is not a token: it is a
hardcoded sentinel inside Chakra's own `defaultConfig.globalCss`, referenced
as `var(--chakra-empty, )` to give the eighteen filter and backdrop-filter
components an empty default. It is never defined anywhere, so it always
resolves to the empty fallback, and both systems emit the identical inert
reference.


In [ ]:
with k.section('isolation'):
    with k.check('jaen emits no --chakra- variables beyond the empty sentinel') as c:
        leaked = [v for v in FACTS['emittedVars']['chakra'] if v != '--chakra-empty']
        c.expect_true(not leaked, 'leaked: %s' % leaked[:8])

    with k.check('the jaen namespace is populated') as c:
        n = len(FACTS['emittedVars']['jaen'])
        c.note('%d variables' % n)
        c.expect_true(n > 400, '%d variables' % n)

    with k.check('the brand reaches the CMS chrome') as c:
        c.expect_true('--jaen-colors-brand-500' in FACTS['emittedVars']['jaen'])


## Colour-mode wiring

Source-level checks: whether the runtime actually flips is a browser
question and belongs to the build notebook.

Nothing synced before this. `gatsby-ssr` rendered `<ColorModeScript
initialColorMode={theme.config.initialColorMode}/>`, jaen's theme set no
`config`, so the emitted script's mode was the literal `"light"`. The site's
own `initialColorMode: 'system'` was read by v2's ColorModeProvider, and the
only provider in the tree was jaen's. The site's source already suspected it:
`//? This doesnt sync`.


In [ ]:
import os, re

def repo_text(rel):
    p = k.repo_path(rel)
    return open(p, encoding='utf-8').read() if os.path.isfile(p) else ''

def code_only(src):
    """Source with comments and string literals removed.

    Grepping raw source for an API name is fragile exactly when the source
    explains itself well: the first version of this notebook reported three
    failures that were all its own prose, because the comments explaining why
    cssVarsRoot and setPreBodyComponents are gone naturally mention them.
    String literals go too, since a message split across a concatenation is
    not a reliable thing to grep for either.
    """
    src = re.sub(r"/\*.*?\*/", " ", src, flags=re.S)
    src = re.sub(r"//[^\n]*", " ", src)
    src = re.sub(r"'(?:[^'\\\n]|\\.)*'", "''", src)
    src = re.sub(r'"(?:[^"\\\n]|\\.)*"', '""', src)
    return src

def message_text(src):
    """Every string literal joined, so a concatenated message greps as one."""
    return " ".join(re.findall(r"'([^'\\\n]*)'", src))

with k.section('colour mode'):
    ssr = repo_text('packages/gatsby-plugin-jaen/gatsby-ssr.tsx')

    with k.check('ColorModeScript is gone (v3 has no such export)') as c:
        c.expect_not_contains(ssr, 'ColorModeScript')

    with k.check('the no-flash script matches next-themes storage contract') as c:
        c.expect_contains(ssr, "localStorage.getItem('theme')")
        c.expect_contains(ssr, 'prefers-color-scheme: dark')
        c.expect_contains(ssr, 'colorScheme')

    with k.check('it runs in head, before first paint') as c:
        ssr_code = code_only(ssr)
        c.expect_contains(ssr_code, 'setHeadComponents')
        c.expect_not_contains(ssr_code, 'setPreBodyComponents')

    root = repo_text('packages/gatsby-plugin-jaen/src/gatsby/wrap-root-element.tsx')

    with k.check('next-themes wraps Chakra, not the other way round') as c:
        c.expect_contains(root, 'NextThemeProvider')
        c.expect_true(root.index('<NextThemeProvider') < root.index('<ChakraProvider'),
                      'the class-setter must enclose the token-reader')

    with k.check('cssVarsRoot is gone') as c:
        c.expect_not_contains(code_only(root), 'cssVarsRoot')

    hook = repo_text('packages/jaen/src/hooks/use-color-mode.tsx')

    with k.check('the four v2 names survive the swap in one file') as c:
        for name in ['useColorMode', 'useColorModeValue', 'DarkMode', 'LightMode']:
            c.expect_contains(hook, name)
        c.expect_contains(hook, "from 'next-themes'")

    with k.check('DarkMode/LightMode stay out of layout') as c:
        c.expect_contains(hook, "display=\"contents\"")
        c.expect_contains(hook, 'hasBackground={false}')


## The shadow contract

A consuming site hands jaen its brand palette by shadowing
`src/gatsby-plugin-jaen/theme/theme.ts`. In v3 that file must export a
`SystemContext`, and jaen asserts it with `isValidSystem`.

The reason is that the config-shaped alternative cannot fail loudly.
`mergeConfigs` is a permissive deep merge, so a leftover v2 `extendTheme()`
result has every key at the wrong nesting level, gets dropped without a word,
and the site ships correct in every respect except that everything branded is
pink.

The path must not change either: renaming it would let a stale shadow orphan
itself silently instead of failing the build.


In [ ]:
with k.section('shadow contract'):
    idx = repo_text('packages/gatsby-plugin-jaen/src/theme/index.ts')

    with k.check('the shadow is validated, not trusted') as c:
        c.expect_contains(idx, 'isValidSystem')
        c.expect_contains(idx, 'throw new Error')

    with k.check('a missing brand palette is its own error') as c:
        c.expect_contains(message_text(idx), 'theme.tokens.colors.brand')

    with k.check('the shadow path is unchanged') as c:
        c.expect_true(os.path.isfile(
            k.repo_path('packages/gatsby-plugin-jaen/src/theme/theme.ts')))

    with k.check('the v2 theme tree is gone') as c:
        c.expect_true(not os.path.isdir(
            k.repo_path('packages/gatsby-plugin-jaen/src/theme/jaen-theme')))


In [ ]:
k.summary()
k.save_results('results-08-theme.json')
rc = k.verdict()
assert rc == 0, 'run has FAILures — see the summary above'
